# 4. Feature Engineering

## Objective

This notebook creates additional behavioral and financial features
from the customer's six-month credit history.

The objective is to capture repayment behavior, payment consistency,
credit utilization, and historical debt patterns that may improve
default prediction.

Only information available before the target month is used in order
to avoid data leakage.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/credit_default_clean.csv"
)

df.shape

(30000, 24)

In [2]:
feature_df = df.copy()

In [3]:
pay_status_cols = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

bill_cols = [
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6"
]

payment_cols = [
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

In [4]:
feature_df["NUM_DELAYED_MONTHS"] = (
    feature_df[pay_status_cols] > 0
).sum(axis=1)

In [5]:
feature_df["MAX_PAYMENT_DELAY"] = (
    feature_df[pay_status_cols]
    .clip(lower=0)
    .max(axis=1)
)

In [6]:
feature_df["HAS_PAYMENT_DELAY"] = (
    feature_df["NUM_DELAYED_MONTHS"] > 0
).astype(int)

In [7]:
feature_df["RECENT_DELAY"] = (
    feature_df["PAY_0"] > 0
).astype(int)

In [8]:
feature_df["AVG_BILL_AMT"] = (
    feature_df[bill_cols]
    .mean(axis=1)
)

In [9]:
feature_df["MAX_BILL_AMT"] = (
    feature_df[bill_cols]
    .max(axis=1)
)

In [10]:
feature_df["AVG_PAY_AMT"] = (
    feature_df[payment_cols]
    .mean(axis=1)
)

In [11]:
feature_df["TOTAL_PAY_AMT"] = (
    feature_df[payment_cols]
    .sum(axis=1)
)

In [12]:
feature_df["ZERO_PAYMENT_MONTHS"] = (
    feature_df[payment_cols] == 0
).sum(axis=1)

In [13]:
feature_df["CREDIT_UTILIZATION"] = (
    feature_df["BILL_AMT1"]
    / feature_df["LIMIT_BAL"]
)

In [14]:
feature_df["AVG_CREDIT_UTILIZATION"] = (
    feature_df["AVG_BILL_AMT"]
    / feature_df["LIMIT_BAL"]
)

In [15]:
feature_df["PAYMENT_TO_BILL_RATIO"] = (
    feature_df["AVG_PAY_AMT"]
    / feature_df["AVG_BILL_AMT"].replace(0, np.nan)
)

In [16]:
feature_df["PAYMENT_TO_BILL_RATIO"].isnull().sum()

np.int64(870)

In [17]:
feature_df["PAYMENT_TO_BILL_RATIO"] = (
    feature_df["PAYMENT_TO_BILL_RATIO"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [18]:
feature_df["BILL_CHANGE"] = (
    feature_df["BILL_AMT1"]
    - feature_df["BILL_AMT6"]
)

In [19]:
new_features = [
    "NUM_DELAYED_MONTHS",
    "MAX_PAYMENT_DELAY",
    "HAS_PAYMENT_DELAY",
    "RECENT_DELAY",
    "AVG_BILL_AMT",
    "MAX_BILL_AMT",
    "AVG_PAY_AMT",
    "TOTAL_PAY_AMT",
    "ZERO_PAYMENT_MONTHS",
    "CREDIT_UTILIZATION",
    "AVG_CREDIT_UTILIZATION",
    "PAYMENT_TO_BILL_RATIO",
    "BILL_CHANGE"
]

feature_df[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
NUM_DELAYED_MONTHS,30000.0,0.834200,1.554303,0.000000,0.000000,0.000000,1.000000,6.000000e+00
MAX_PAYMENT_DELAY,30000.0,0.682200,1.073518,0.000000,0.000000,0.000000,2.000000,8.000000e+00
HAS_PAYMENT_DELAY,30000.0,0.335633,0.472219,0.000000,0.000000,0.000000,1.000000,1.000000e+00
RECENT_DELAY,30000.0,0.227267,0.419073,0.000000,0.000000,0.000000,0.000000,1.000000e+00
AVG_BILL_AMT,30000.0,44976.945200,63260.721860,-56043.166667,4781.333333,21051.833333,57104.416667,8.773138e+05
MAX_BILL_AMT,30000.0,60572.436867,78404.814025,-6029.000000,10060.000000,31208.500000,79599.000000,1.664089e+06
AVG_PAY_AMT,30000.0,5275.232094,10137.946323,0.000000,1113.291667,2397.166667,5583.916667,6.273443e+05
TOTAL_PAY_AMT,30000.0,31651.392567,60827.677939,0.000000,6679.750000,14383.000000,33503.500000,3.764066e+06
ZERO_PAYMENT_MONTHS,30000.0,1.229900,1.718588,0.000000,0.000000,0.000000,2.000000,6.000000e+00
CREDIT_UTILIZATION,30000.0,0.423771,0.411462,-0.619892,0.022032,0.313994,0.829843,6.455300e+00


In [20]:
feature_df[new_features].isnull().sum()

NUM_DELAYED_MONTHS        0
MAX_PAYMENT_DELAY         0
HAS_PAYMENT_DELAY         0
RECENT_DELAY              0
AVG_BILL_AMT              0
MAX_BILL_AMT              0
AVG_PAY_AMT               0
TOTAL_PAY_AMT             0
ZERO_PAYMENT_MONTHS       0
CREDIT_UTILIZATION        0
AVG_CREDIT_UTILIZATION    0
PAYMENT_TO_BILL_RATIO     0
BILL_CHANGE               0
dtype: int64

In [21]:
np.isinf(
    feature_df[new_features]
    .select_dtypes(include=np.number)
).sum()

NUM_DELAYED_MONTHS        0
MAX_PAYMENT_DELAY         0
HAS_PAYMENT_DELAY         0
RECENT_DELAY              0
AVG_BILL_AMT              0
MAX_BILL_AMT              0
AVG_PAY_AMT               0
TOTAL_PAY_AMT             0
ZERO_PAYMENT_MONTHS       0
CREDIT_UTILIZATION        0
AVG_CREDIT_UTILIZATION    0
PAYMENT_TO_BILL_RATIO     0
BILL_CHANGE               0
dtype: int64

In [22]:
print("Original shape:", df.shape)
print("Engineered shape:", feature_df.shape)

Original shape: (30000, 24)
Engineered shape: (30000, 37)


## Data Leakage Check

All engineered features are created exclusively from customer information
and historical financial behavior observed before the target month.

The target variable `DEFAULT` is not used to construct any input feature.

Therefore, no target leakage is introduced during feature engineering.

## Feature Engineering Summary

New behavioral and financial features were created from the six-month
customer history.

### Repayment Behavior
- `NUM_DELAYED_MONTHS`
- `MAX_PAYMENT_DELAY`
- `HAS_PAYMENT_DELAY`
- `RECENT_DELAY`
- `ZERO_PAYMENT_MONTHS`

### Financial Behavior
- `AVG_BILL_AMT`
- `MAX_BILL_AMT`
- `AVG_PAY_AMT`
- `TOTAL_PAY_AMT`
- `CREDIT_UTILIZATION`
- `AVG_CREDIT_UTILIZATION`
- `PAYMENT_TO_BILL_RATIO`
- `BILL_CHANGE`

The original variables are preserved alongside the engineered features.

No information from the target variable was used to construct these features.

In [23]:
feature_df.to_csv(
    "../data/processed/credit_default_features.csv",
    index=False
)

In [24]:
feature_df.shape

(30000, 37)

In [25]:
feature_df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,RECENT_DELAY,AVG_BILL_AMT,MAX_BILL_AMT,AVG_PAY_AMT,TOTAL_PAY_AMT,ZERO_PAYMENT_MONTHS,CREDIT_UTILIZATION,AVG_CREDIT_UTILIZATION,PAYMENT_TO_BILL_RATIO,BILL_CHANGE
0,20000,2,2,1,24,2,2,-1,-1,-2,...,1,1284.000000,3913,114.833333,689,5,0.195650,0.064200,0.089434,3913
1,120000,2,2,2,26,-1,2,0,0,0,...,0,2846.166667,3455,833.333333,5000,2,0.022350,0.023718,0.292791,-579
2,90000,2,2,2,34,0,0,0,0,0,...,0,16942.166667,29239,1836.333333,11018,0,0.324878,0.188246,0.108388,13690
3,50000,2,2,1,37,0,0,0,0,0,...,0,38555.666667,49291,1398.000000,8388,0,0.939800,0.771113,0.036259,17443
4,50000,1,2,1,57,-1,0,-1,0,0,...,0,18223.166667,35835,9841.500000,59049,0,0.172340,0.364463,0.540054,-10514
